# Masking into MERRA-2 Datasets

A little notebook to showcase how I might start using my catalog to mask into MERRA-2

In [1]:
import pandas as pd
import xarray as xr
import numpy as np
import os
from pathlib import Path
import seaborn as sns
import xarray as xr
from tqdm import tqdm
import dask

import h5py
import boto3
import s3fs
import earthaccess
from IPython.display import display, Markdown
from ipywidgets import IntProgress

home_dir = str(Path(os.getcwd()).parents[0])

## Setting Up MERRA-2 Streaming

In [2]:
if (boto3.client('s3').meta.region_name == 'us-west-2'):
    display(Markdown('### us-west-2 Region Check: &#x2705;'))
else:
    display(Markdown('### us-west-2 Region Check: &#10060;'))
    raise ValueError('Your notebook is not running inside the AWS us-west-2 region, and will not be able to directly access NASA Earthdata S3 buckets')

### us-west-2 Region Check: &#x2705;

## Loading Up the Catalog

In [3]:
# load up all of the dataframes by year, and then concatenate into one big one
df_path = home_dir + '/data/ar_database/dataframes/'
fnames = os.listdir(df_path)
df_list = []

for fname in fnames:
    df_list.append(pd.read_hdf(df_path + fname))
    
dataframe = pd.concat(df_list)

## Storm Characteristics

Adding storm characteristics to the database using MERRA-2 data.

### Areas and Durations

In [ ]:
# load up relevant MERRA-2 datasets
grid_areas = xr.open_dataset(home_dir + '/data/area/MERRA2_gridarea.nc')
grid_areas = grid_areas.sel(lat=slice(-86, -39)).cell_area

In [ ]:
def compute_max_area(ar_da):
    da = ar_da.rename({'lats':'lat', 'lons':'lon'})
    grid_area_storm = grid_areas.sel(lat=da.lat, lon=da.lon)
    max_area = float(da.dot(grid_area_storm).max().values/(1000**2))
    return max_area

def compute_mean_area(ar_da):
    da = ar_da.rename({'lats':'lat', 'lons':'lon'})
    grid_area_storm = grid_areas.sel(lat=da.lat, lon=da.lon)
    mean_area = float(da.dot(grid_area_storm).mean().values/(1000**2))
    return mean_area

def compute_duration(ar_da):
    days = (ar_da.time.max() - ar_da.time.min()).values.astype('timedelta64[h]').astype(int) + np.timedelta64(3, 'h')
    return days

def add_start_date(ar_da):
    start = ar_da.time.min().values
    return start

def add_end_date(ar_da):
    end = ar_da.time.max().values
    return end

In [ ]:
dataframe['max_area'] = dataframe['data_array'].apply(compute_max_area)
dataframe['mean_area'] = dataframe['data_array'].apply(compute_mean_area)
dataframe['duration'] = dataframe['data_array'].apply(compute_duration)
dataframe['start_date'] = dataframe['data_array'].apply(add_start_date)
dataframe['end_date'] = dataframe['data_array'].apply(add_end_date)

In [ ]:
attr_df = dataframe[['max_area', 'mean_area', 'duration', 'start_date', 'end_date', 'is_landfalling']]

In [ ]:
attr_df = attr_df.sort_values(by='start_date')

### Atmospheric Variables

In [ ]:
cell_areas = xr.open_dataset('/home/jovyan/extreme_antarctic_ARs/data/area/MERRA2_gridarea.nc')
cell_areas = cell_areas.cell_area
ais_mask = xr.open_dataset('/home/jovyan/extreme_antarctic_ARs/data/antarctic_masks/AIS_Full_basins_Zwally_MERRA2grid_new.nc')
ais_mask = ais_mask > 0

In [20]:
def compute_cumulative(storm_da, var_da, area_da, ais_da=None):
    if ais_da:
        storm_ais_mask = ais_da.sel(lat=storm_da.lat, lon=storm_da.lon).Zwallybasins
        storm_da_subset = storm_da.where(storm_ais_mask, 0)
    else:
        storm_da_subset = storm_da.copy()

    var_da_subset = var_da.sel(lat=storm_da_subset.lat, lon=storm_da_subset.lon)

    storm_cell_areas = area_da.sel(lat=storm_da.lat, lon=storm_da.lon)
    avg_storm_val = storm_cell_areas.dot(storm_da_subset*var_da_subset).mean().values

    return avg_storm_val

def compute_max_intensity(storm_da, var_da, area_da, ais_da=None):
    if ais_da:
        storm_ais_mask = ais_da.sel(lat=storm_da.lat, lon=storm_da.lon).Zwallybasins
        storm_da_subset = storm_da.where(storm_ais_mask, 0)
    else:
        storm_da_subset = storm_da.copy()
        
    var_da_subset = var_da.sel(lat=storm_da_subset.lat, lon=storm_da_subset.lon)
    max_intensity_val = (storm_da_subset*var_da_subset).max().values

    return max_intensity_val

def compute_average(storm_da, var_da, area_da, ais_da=None):
    if ais_da:
        storm_ais_mask = ais_da.sel(lat=storm_da.lat, lon=storm_da.lon).Zwallybasins
        storm_da_subset = storm_da.where(storm_ais_mask, 0)
    else:
        storm_da_subset = storm_da.copy()

    var_da_subset = var_da.sel(lat=storm_da_subset.lat, lon=storm_da_subset.lon)

    storm_cell_areas = area_da.sel(lat=storm_da.lat, lon=storm_da.lon)
    tot_area = storm_da_subset.dot(storm_cell_areas)
    avg_storm_val = (storm_cell_areas.dot(storm_da_subset*var_da_subset)/tot_area).mean().values

    return avg_storm_val

#### 2m-Temperature, 10m Poleward Wind, Total Precipitable Vapor

In [4]:
data_doi = '10.5067/3Z173KIE2TPD'
vars = ['T2M', 'V10M', 'SLP']

In [ ]:
%%time
auth = earthaccess.login()

In [ ]:
%%time

i = 15

storm_da = dataframe.iloc[i].data_array
first = np.min(storm_da.time.dt.date.to_numpy())
last = np.max(storm_da.time.dt.date.to_numpy())

In [ ]:
%%time
results = earthaccess.search_data(doi=data_doi, 
                                  temporal=(f'{first.year}-{first.month}-{first.day}', 
                                            f'{last.year}-{last.month}-{last.day}'));

In [ ]:
%%time
obs_ds = xr.open_mfdataset(earthaccess.open(results));
obs_ds = obs_ds.sel(time = obs_ds.time.dt.hour % 3 == 0).sel(lat = slice(-86, -39))
obs_ds = obs_ds[vars]

In [ ]:
%%time
compute_average(storm_da, obs_ds.V10M, cell_areas)

In [21]:
def compute_summary(storm_da, func, var, cell_areas, data_doi):
    first = np.min(storm_da.time.dt.date.to_numpy())
    last = np.max(storm_da.time.dt.date.to_numpy())
    # stream the data only between those two dates
    results = earthaccess.search_data(doi=data_doi, 
                                  temporal=(f'{first.year}-{first.month}-{first.day}', 
                                            f'{last.year}-{last.month}-{last.day}'))

    obs_ds = xr.open_mfdataset(earthaccess.open(results));
    obs_ds = obs_ds.sel(time = obs_ds.time.dt.hour % 3 == 0).sel(lat = slice(-86, -39))
    obs_da = obs_ds[var]

    summary = func(storm_da, obs_da, cell_areas)

    return summary
    

In [22]:
from dask.distributed import Client, progress
client = Client(threads_per_worker=4, n_workers=3)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/jbbutler/proxy/40629/status,
Dashboard: /user/jbbutler/proxy/40629/status,Workers: 3
Total threads: 12,Total memory: 14.84 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:45751,Workers: 3
Dashboard: /user/jbbutler/proxy/40629/status,Total threads: 12
Started: Just now,Total memory: 14.84 GiB
Comm: tcp://127.0.0.1:44575,Total threads: 4
Dashboard: /user/jbbutler/proxy/42925/status,Memory: 4.95 GiB
Nanny: tcp://127.0.0.1:45933,


In [25]:
sub_dataframe = dataframe.iloc[0:20]
lazy_results = []

for index, storm in sub_dataframe.iterrows():
    
    cell_areas = dask.delayed(lambda fname: xr.open_dataset(fname).cell_area)('/home/jovyan/extreme_antarctic_ARs/data/area/MERRA2_gridarea.nc')
    lazy_result = dask.delayed(compute_summary)(storm.data_array, compute_average, cell_areas, 'T2M', '10.5067/3Z173KIE2TPD')
    lazy_results.append(lazy_result)


In [26]:
import warnings
warnings.filterwarnings('ignore')

results = dask.compute(*lazy_results)

QUEUEING TASKS | : 100%|██████████| 1/1 [00:00<00:00, 3006.67it/s]
QUEUEING TASKS | : 100%|██████████| 2/2 [00:00<00:00, 1178.01it/s]
PROCESSING TASKS | : 100%|██████████| 2/2 [00:00<00:00, 35.65it/s]
COLLECTING RESULTS | : 100%|██████████| 2/2 [00:00<00:00, 66052.03it/s]

QUEUEING TASKS | : 100%|██████████| 3/3 [00:00<00:00, 3581.81it/s]

QUEUEING TASKS | : 100%|██████████| 2/2 [00:00<00:00, 2915.75it/s]
QUEUEING TASKS | : 100%|██████████| 1/1 [00:00<00:00, 2716.52it/s]
PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | : 100%|██████████| 2/2 [00:00<00:00, 2881.69it/s]


PROCESSING TASKS | :   0%|          | 0/2 [00:00<?, ?it/s]


QUEUEING TASKS | : 100%|██████████| 1/1 [00:00<00:00, 2407.75it/s]



PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]
QUEUEING TASKS | : 100%|██████████| 1/1 [00:00<00:00, 2432.89it/s]

PROCESSING TASKS | : 100%|██████████| 1/1 [00:00<00:00, 22.53it/s]
COLLECTING RESULTS | : 100%|██████████| 1/1 [00:00<00:00, 21509.25it/s

ValueError: Unsupported key-type <class 'xarray.core.dataarray.DataArray'>

QUEUEING TASKS | : 100%|██████████| 1/1 [00:00<00:00, 2448.51it/s]
PROCESSING TASKS | : 100%|██████████| 1/1 [00:00<00:00, 54.92it/s]
COLLECTING RESULTS | : 100%|██████████| 1/1 [00:00<00:00, 19878.22it/s]


In [ ]:
results

In [ ]:
auth = earthaccess.login()

data_doi = '10.5067/3Z173KIE2TPD'
vars = ['T2M']

dataframe_landfalling = dataframe[dataframe.is_landfalling]
t2m_vals = [None]*len(dataframe_landfalling)

for i in tqdm(range(len(dataframe_landfalling))):

    storm_da = dataframe.iloc[i].data_array
    storm_da = storm_da.rename({'lats':'lat', 'lons':'lon'})

    first = np.min(storm_da.time.dt.date.to_numpy())
    last = np.max(storm_da.time.dt.date.to_numpy())
    # stream the data only between those two dates
    results = earthaccess.search_data(doi=data_doi, 
                                  temporal=(f'{first.year}-{first.month}-{first.day}', 
                                            f'{last.year}-{last.month}-{last.day}'));
    obs_ds = xr.open_mfdataset(earthaccess.open(results));
    obs_ds = obs_ds.sel(time = obs_ds.time.dt.hour % 3 == 0).sel(lat = slice(-86, -39))
    obs_ds = obs_ds[vars]

    t2m_vals[i] = compute_average(storm_da, obs_ds.T2M, cell_areas, ais_mask)

In [ ]:
first = np.min(storm_da.time.dt.date.to_numpy())
last = np.max(storm_da.time.dt.date.to_numpy())
# stream the data only between those two dates
results = earthaccess.search_data(doi=data_doi, 
                                  temporal=(f'{first.year}-{first.month}-{first.day}', 
                                            f'{last.year}-{last.month}-{last.day}'))
obs_ds = xr.open_mfdataset(earthaccess.open(results))
obs_ds = obs_ds.sel(time = obs_ds.time.dt.hour % 3 == 0).sel(lat = slice(-86, -39))
obs_ds = obs_ds[vars]
obs_ds = obs_ds.sel(lat=storm_da.lat, lon=storm_da.lon)

In [ ]:
sns.scatterplot(data=attr_df, x='start_date', y='mean_area', alpha=0.1)

In [ ]:
sns.scatterplot(data=attr_df[attr_df['is_landfalling']], x='start_date', y='mean_area', alpha=0.1)

In [ ]:
sns.scatterplot(data=attr_df[attr_df['is_landfalling']], x='start_date', y='max_area', alpha=0.1);

In [ ]:
attr_df[attr_df.max_area == attr_df.max_area.max()]

In [ ]:
sns.histplot(data=attr_df, x='max_area', bins=10)

In [ ]:
sns.histplot(data=attr_df, x='start_date', bins=20)

In [ ]:
sns.histplot(data=attr_df[attr_df['is_landfalling']], x='start_date', bins=20)

### 10m Winds

In [ ]:
start_day = attr_df.start_date.dt.date.iloc[0]
end_day = attr_df.end_date.dt.date.iloc[-1]

In [ ]:
days = pd.date_range(start=start_day, end=end_day).strftime('%Y%m%d')

In [ ]:
days

In [ ]:
# load up the 10m wind data

day = days[15000]

In [ ]:
day

In [ ]:
path

In [ ]:
path = 's3://gesdisc-cumulus-prod-protected/MERRA2/M2I1NXASM.5.12.4/'+ day[0:4] +'/' + day[4:6] + '/MERRA2_400.inst1_2d_asm_Nx.' + day + '.nc4'
#path = 's3://gesdisc-cumulus-prod-protected/MERRA2/M2I1NXASM.5.12.4/2022/03/MERRA2_400.inst1_2d_asm_Nx.20220315.nc4'
dat = xr.open_dataset(fs.open(path), decode_cf=True,)

In [ ]:

# Files are organized by s3://gesdisc-cumulus-prod-protected/MERRA2/M2T1NXSLV.5.12.4/year/mo/*.nc4
datasets = []

for day in days:
    
    path = 's3://gesdisc-cumulus-prod-protected/MERRA2/M2I1NXASM.5.12.4/'+ day[0:4] +'/' + day[4:6] + '/MERRA2_400.inst1_2d_asm_Nx.' + day + '.nc4'
    dat = xr.open_dataset(fs.open(path), decode_cf=True,)
    antarctic_dat = dat.sel(lat = slice(-90.0, -50.0), lon = slice(-180, 179.4))
    datasets.append(antarctic_dat[['T2M', 'TQV', 'V10M']])

#### h5coro implementation

In [ ]:
# tried to use h5coro at one point to load up the data granules, instead of using xarray openmfdataset
# seems like it's not much faster at all, and while it could be if I could input multiple days, documentation for the package is very bad
# and I don't feel like reading source code to find a magical function that optimizes the loading of multiple days of data..

# Initialize the H5Coro object with the S3 driver and credentials
#h5obj = h5coro.H5Coro(paths[0].details['name'], h5coro.s3driver.S3Driver, 
                      credentials=s3_creds, errorChecking=True, verbose=False, multiProcess=False)

# Define the variables you want to read
#variables = ['SLP', 'T2M', 'V10M']
# Read the data using h5coro
#data = h5obj.readDatasets(variables, block=True, enableAttributes=False)

#time = pd.date_range(start=pd.Timestamp(start_date), end=pd.Timestamp(end_date)+pd.Timedelta(1, unit='D'), freq='1h', inclusive='left')
#lat = np.arange(-90, 90.5, .5)
#lon = np.arange(-180, 180, 0.625)